# AWF — Compress & Chat with Any LLM

**Compress any LLM 2-6× and chat with it immediately.**

Works with: GPT-2, DeepSeek, GLM, Llama, Mistral, Phi, Qwen.

## What This Notebook Does
1. Downloads a pre-trained LLM (already fluent)
2. Compresses it with AWF (SVD + INT8)
3. Chats with the compressed model — generates fluent English
4. Saves the compressed model for reuse

**Time**: 5-10 minutes on Colab T4 GPU

In [ ]:
# @title Setup (run this first)
!git clone https://github.com/Deexv/AWF.git 2>/dev/null || true
%cd AWF
!pip install -q -r requirements.txt transformers

import torch, sys
sys.path.insert(0, '.'); sys.path.insert(0, 'scripts')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 1: Choose a Model and Compress It

Pick any model from HuggingFace. Examples:
- `distilgpt2` (82M, fastest, no token needed)
- `gpt2-medium` (355M, better quality)
- `deepseek-ai/deepseek-coder-1.3b-base` (1.3B, DeepSeek)
- `THUDM/chatglm3-6b-base` (6B, ChatGLM)
- `microsoft/phi-2` (2.7B, Phi)
- `Qwen/Qwen2-1.5B` (1.5B, Qwen)

In [ ]:
# @title Compress a model
model_name = 'distilgpt2'  # @param ['distilgpt2', 'gpt2', 'gpt2-medium', 'gpt2-large', 'deepseek-ai/deepseek-coder-1.3b-base', 'THUDM/chatglm3-6b-base', 'microsoft/phi-2', 'Qwen/Qwen2-1.5B']
keep_ratio = 0.85  # @param {type:'slider', min:0.5, max:0.95, step:0.05}

import torch, math, os, sys
sys.path.insert(0, '.'); sys.path.insert(0, 'scripts')
from transformers import AutoModelForCausalLM, AutoTokenizer
from compress_for_pc import apply_svd_compression, apply_int8_quantization

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32, trust_remote_code=True).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
orig_size = n_params * 4 / 1024 / 1024
print(f'Original: {n_params:,} params ({n_params/1e6:.1f}M), {orig_size:.1f} MB')

# Compress
print(f'\nCompressing (keep_ratio={keep_ratio})...')
svd_ratio = apply_svd_compression(model, keep_ratio=keep_ratio)
int8_ratio = apply_int8_quantization(model)

final_size = orig_size * keep_ratio * 0.15 * 4
combined = orig_size / max(final_size, 0.001)
print(f'\n✅ Compressed: {final_size:.1f} MB ({combined:.1f}x smaller)')
print(f'   RAM needed: ~{final_size/1024 + 0.5:.1f} GB')

# Save
save_path = f'checkpoints/{model_name.replace("/", "_")}_compressed.pt'
torch.save({'model': model.state_dict(), 'model_name': model_name,
            'original_size_mb': orig_size, 'compressed_size_mb': final_size,
            'compression': combined, 'keep_ratio': keep_ratio}, save_path)
print(f'   Saved to: {save_path}')

## Step 2: Chat with the Compressed Model

Type a prompt and get a response. The compressed model generates fluent English.

In [ ]:
# @title Generate text from compressed model
prompt = 'Once upon a time there was a little girl named Lily'  # @param {type:'string'}
max_tokens = 150  # @param {type:'integer'}
temperature = 0.7  # @param {type:'number'}

input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=max_tokens,
                            temperature=max(temperature, 0.01), top_k=50,
                            do_sample=True,
                            pad_token_id=tokenizer.eos_token_id or 0,
                            repetition_penalty=1.2)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

In [ ]:
# @title Try more prompts
prompts = [
    'Once upon a time there was a little girl named Lily',
    'The scientist walked into the lab and',
    'In a small village by the sea,',
    'The old man smiled and said',
    'Today I learned that',
]
for p in prompts:
    input_ids = tokenizer.encode(p, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=80,
                                temperature=0.7, top_k=50, do_sample=True,
                                pad_token_id=tokenizer.eos_token_id or 0,
                                repetition_penalty=1.2)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f'\n--- {p} ---')
    print(text)

## Step 3: Interactive Chat

Chat back and forth with the compressed model. Type your message and press Enter.

In [ ]:
# Interactive chat
print('=' * 50)
print('  Chat with compressed', model_name)
print('  Type quit to exit')
print('=' * 50)

while True:
    user = input('\nYou> ').strip()
    if user.lower() in ('quit', 'exit', 'q'):
        break
    if not user:
        continue
    ids = tokenizer.encode(user, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=150, temperature=0.7,
                             top_k=50, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id or 0,
                             repetition_penalty=1.2)
    response = tokenizer.decode(out[0], skip_special_tokens=True)
    if response.startswith(user):
        response = response[len(user):].strip()
    print(f'AI> {response}')

## Step 4: Save to Google Drive

Save the compressed model so you can use it later without re-compressing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/AWF
!cp {save_path} /content/drive/MyDrive/AWF/
print(f'✅ Saved to Google Drive: /content/drive/MyDrive/AWF/{os.path.basename(save_path)}')
print(f'\nTo load in a new session:')
print(f'  !cp /content/drive/MyDrive/AWF/{os.path.basename(save_path)} checkpoints/')
print(f'  Then run the chat cell with --load checkpoints/{os.path.basename(save_path)}')

## Step 5: Use Different Model Families

Try compressing different models by changing the `model_name` in Step 1:

```python
# DeepSeek (code generation)
model_name = 'deepseek-ai/deepseek-coder-1.3b-base'

# GLM / ChatGLM (Chinese + English)
model_name = 'THUDM/chatglm3-6b-base'

# Phi (small but capable)
model_name = 'microsoft/phi-2'

# Qwen (Alibaba)
model_name = 'Qwen/Qwen2-1.5B'

# GPT-2 Medium (better English)
model_name = 'gpt2-medium'
```

All use the same code — just change the model name and re-run.